# Check an Analyst report

Copy this file into `private/reviews/` and rename the copy to `<packet_id>_review.ipynb`
(for example, `EP_0003_review.ipynb`). Work in that copy; keep this template blank.

1. Set `attempt_id` below to the folder name printed by generation.
2. Run the cells and read each claim beside its evidence.
3. Fill in **Notes / feedback** and make any fixes in the edit cell.
4. When finished, set `ready = True`, run all cells again, and save the notebook.

The last cell saves the final report to public. The original candidate stays unchanged.
There are no model calls here. Leave `ready = False` while checking.


In [6]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "046ada01cdfd4b519f7aeca2487d7199"  # change this in your copy
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0013 | Model: google/gemini-2.5-flash
Packet masking changed: False


In [7]:
print("Recommendation:", report["recommendation"])
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])

Recommendation: HOLD

CLAIM C1 Net sales increased by $245 million, or 0.29%, from $84,039 million in fiscal year 2024 to $84,284 million in fiscal year 2025 (Item 7, p.19). This modest growth was primarily driven by a 1% increase in pricing, largely offset by unfavorable foreign exchange impacts of 1% (Item 7, p.21).
EVIDENCE EP_0013_E01 SUMMARY OF 2025 RESULTS
Amounts in millions, except per share amounts 2025 2024 Change vs. Prior Year
Net sales $ 84,284 $ 84,039 — %
Operating income 20,451 18,545 10 %
Net earnings 16,065 14,974 7 %
Net earnings attributable to COMPANY_13 15,974 14,879 7 %
Diluted net earnings per common share 6.51 6.02 8 %
Core earnings per share 6.83 6.59 4 %
Cash flow from operating activities 17,817 19,846 (10)%
• Net sales increased $245 million to $84.3 billion versus the prior year. Net sales increased low single digits in SEGMENT_28 and was unchanged in
SEGMENT_29, SEGMENT_30 and SEGMENT_31. Net Sales declined low single digits in SEGMENT_32. Organic sales, 

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [8]:
facts_by_id = {f["fact_id"]: f for f in packet["facts"]}
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    ids = [spec.get("fact_id")] if spec["kind"] == "numeric" else [spec.get("from_fact_id"), spec.get("to_fact_id")]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    if all(ids):
        selected = [facts_by_id.get(fact_id) for fact_id in ids]
        print(claim["claim_id"], "Referenced facts:")
        pprint(selected)
        if any(f is None for f in selected):
            print("Unknown fact ID; check manually.")
            continue
    else:
        print(claim["claim_id"], "Older candidate: references are added in the edit cell below.")
        matches = [[f for f in packet["facts"] if f["metric"] == spec["metric"]
                    and f["evidence_id"] in claim["evidence_ids"] and f["period"] == period
                    and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])] for period in periods]
        if any(len(found) != 1 for found in matches):
            print("No exact fact match; check manually.")
            continue
        selected = [found[0] for found in matches]
        pprint(selected)
    values = [Decimal(f["value"]) for f in selected]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))


C1 Referenced facts:
[{'evidence_id': 'EP_0013_E01',
  'fact_id': 'EP_0013_F02',
  'metric': 'net_sales',
  'period': {'end': '2024-06-30', 'label': 'FY2024', 'start': '2023-07-01'},
  'unit': 'USD_million',
  'value': '84039'},
 {'evidence_id': 'EP_0013_E01',
  'fact_id': 'EP_0013_F01',
  'metric': 'net_sales',
  'period': {'end': '2025-06-30', 'label': 'FY2025', 'start': '2024-07-01'},
  'unit': 'USD_million',
  'value': '84284'}]
C1 claimed: 0.29 calculated: 0.2915313128428467735218172515 rounded: 0.29
C2 Referenced facts:
[{'evidence_id': 'EP_0013_E01',
  'fact_id': 'EP_0013_F04',
  'metric': 'operating_income',
  'period': {'end': '2024-06-30', 'label': 'FY2024', 'start': '2023-07-01'},
  'unit': 'USD_million',
  'value': '18545'},
 {'evidence_id': 'EP_0013_E01',
  'fact_id': 'EP_0013_F03',
  'metric': 'operating_income',
  'period': {'end': '2025-06-30', 'label': 'FY2025', 'start': '2024-07-01'},
  'unit': 'USD_million',
  'value': '20451'}]
C2 claimed: 10.28 calculated: 10.27770

## Notes / feedback

Replace the blanks with short notes. Use the evidence-checking notebook to check the source PDF too.

- Source excerpts and masking: ...
- Claims, numbers, years, units and citations: ...
- Reasoning and risks / counter-evidence: ...
- - Problems found and what I changed (or no changes needed): The figures and calculations were correct. Replaced ambiguous or incorrect page references with the supporting evidence IDs, including the competition-risk citation.
- Decision: done citation corrections


In [9]:
# Make any corrections here. Skip this if no changes are needed.
# Example: report["claims"][0]["text"] = "Corrected claim..."
# If you correct a number, update its check_spec too, if present.
# Keep the edits in this cell so running the notebook again applies them.

for claim in report["claims"]:
    claim["text"] = (
        claim["text"]
        .replace("Item 7, p.19", "Item 7, EP_0013_E01")
        .replace("Item 7, p.21", "Item 7, EP_0013_E02")
    )

report["rationale"] = report["rationale"].replace(
    "Item 1A, p.23", "Item 1A, EP_0013_E03"
)
# Add explicit source-fact references to the reviewed copy of older candidates.
# This does not change the saved model response, numbers, or wording.
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        continue
    fields = [("fact_id", "period")] if spec["kind"] == "numeric" else [("from_fact_id", "from_period"), ("to_fact_id", "to_period")]
    for id_field, period_field in fields:
        if spec.get(id_field):
            continue
        matches = [f for f in packet["facts"] if f["metric"] == spec["metric"]
                   and f["period"] == spec[period_field] and f["evidence_id"] in claim["evidence_ids"]
                   and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])]
        assert len(matches) == 1, "Check the fact mapping manually: " + claim["claim_id"]
        spec[id_field] = matches[0]["fact_id"]

pprint(report, sort_dicts=False)


{'schema_version': '0.1',
 'report_id': 'R_046ada01cdfd4b519f7aeca2487d7199',
 'recommendation': 'HOLD',
 'claims': [{'claim_id': 'C1',
             'text': 'Net sales increased by $245 million, or 0.29%, from '
                     '$84,039 million in fiscal year 2024 to $84,284 million '
                     'in fiscal year 2025 (Item 7, EP_0013_E01). This modest '
                     'growth was primarily driven by a 1% increase in pricing, '
                     'largely offset by unfavorable foreign exchange impacts '
                     'of 1% (Item 7, EP_0013_E02).',
             'evidence_ids': ['EP_0013_E01', 'EP_0013_E02'],
             'check_spec': {'kind': 'percentage_change',
                            'from_fact_id': 'EP_0013_F02',
                            'to_fact_id': 'EP_0013_F01',
                            'metric': 'net_sales',
                            'from_period': {'label': 'FY2024',
                                            'start': '2023-07-01',
  

## Save the final copy

Set `ready = True` only when the checks and edits are finished.
This checks the edited report and saves it with the masked packet to `public/`.
It replaces existing files with the same IDs, so use this for the working dataset.
Keep released dataset versions unchanged.

Save this notebook with its outputs. These notes are also the feedback;
there is no separate review JSON or feedback file to fill in.


The report is saved as `public/reports/<packet_id>/clean.json`.
For older candidates, the edit cell adds fact IDs only when metric, period, unit and evidence match uniquely. Check the printed references before saving.

In [11]:
ready = True
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report, require_fact_ids=True)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Saved: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/public/reports/EP_0013/clean.json
